# ⛅ Weather-Advisory Support Bot
### Interactive Manual Testing & Evaluation Notebook (Google Colab & Local Jupyter)

This notebook provides an end-to-end interactive testbed and formal verification suite for the **Weather-Advisory Support Bot**.

**Key Architectural Guarantees:**
1. **Real LangGraph Directed Graph:** Multi-node graph with conditional branching (not a single prompt-and-response chain).
2. **100% Policy Grounding:** Every safety recommendation maps strictly to an approved Standard Operating Procedure (SOP) from `sops.yaml` cited by ID.
3. **No Hallucinated Weather Values:** All numbers in answers are grounded strictly in live Open-Meteo forecast API responses and verified via a deterministic guardrail.
4. **Deterministic Conflict Resolution:** When multiple SOPs match, one primary leads, secondary policies are surfaced with full advice, and the system explicitly explains **why** the primary rule was selected.
5. **Zero-Code Policy Updates:** SOPs are loaded fresh on every turn. An 11th/17th SOP can be added live without restarting or touching control-flow code.


In [1]:
# Setup & Dependency Installation
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab environment.")
    get_ipython().system("pip install -q langgraph langchain langchain-google-genai google-genai requests pyyaml pydantic python-dotenv")
except ImportError:
    IN_COLAB = False
    print("Running in local environment.")

import os
import sys
import time
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

# Ensure current working directory is in path
sys.path.insert(0, os.path.abspath('.'))

load_dotenv()

# If in Colab and GOOGLE_API_KEY is not in .env, check Colab Secrets
if IN_COLAB and not os.environ.get("GOOGLE_API_KEY"):
    try:
        from google.colab import userdata
        colab_key = userdata.get("GOOGLE_API_KEY")
        if colab_key:
            os.environ["GOOGLE_API_KEY"] = colab_key
            print("Loaded GOOGLE_API_KEY from Google Colab Secrets.")
    except Exception:
        pass

# Ensure default model settings
if not os.environ.get("MODEL_PROVIDER"):
    os.environ["MODEL_PROVIDER"] = "google_genai"
if not os.environ.get("MODEL"):
    os.environ["MODEL"] = "gemini-3.5-flash-lite"

print('Active Model Provider:', os.environ.get('MODEL_PROVIDER'))
print('Active Model:', os.environ.get('MODEL'))


Running in local environment.
Active Model Provider: google_genai
Active Model: gemini-3.5-flash-lite


## 1. Graph Architecture Inspection
Inspect the nodes, conditional edges, and execution flow of the compiled LangGraph agent.


In [ ]:
import graph

# Display graph nodes
print('=== COMPILED LANGGRAPH NODES ===')
for node_name in graph.bot_graph.nodes:
    print(f'  [Node] {node_name}')

print('\nConditional branching sits at:')
print('  - parse_intent     -> (resolve_location | not_in_scope_reply)')
print('  - resolve_location -> (fetch_weather | error_reply)')
print('  - fetch_weather    -> (match_sops | error_reply)')
print('  - match_sops       -> (resolve_conflicts | no_match_reply)')
print('  - resolve_conflicts-> compose_answer -> verify_answer -> END')


## 2. Interactive Manual Query Sandbox
Test any question live with full traceability. Change the `user_query` below and run the cell!


In [ ]:
def test_interactive_query(query: str):
    print(f'User Query: "{query}"\n' + '-'*60)
    res = graph.ask_bot(query)
    
    print('BOT REPLY:')
    print(res.get('reply'))
    print('\n' + '='*60)
    print('AUDIT TRACEABILITY:')
    chosen = res.get('chosen_sop')
    if chosen:
        print(f"  * Primary SOP:     {chosen.get('id')} - {chosen.get('title')} (Severity {chosen.get('severity')})")
    else:
        print("  * Primary SOP:     None (Honest boundary refusal)")
        
    if res.get('selection_reason'):
        print(f"  * Selection Reason: {res.get('selection_reason')}")
        
    others = res.get('other_sops', [])
    if others:
        sec_ids = ', '.join([s['id'] for s in others])
        print(f"  * Secondary SOPs:  {sec_ids}")
        
    facts = res.get('weather_facts')
    if facts:
        print(f"  * Verified Facts:  Temp: {facts.get('temperature_2m')} C | Wind: {facts.get('wind_speed_10m')} km/h | Gusts: {facts.get('wind_gusts_10m')} km/h | Rain: {facts.get('precipitation')} mm ({facts.get('precipitation_probability')}%) | UV: {facts.get('uv_index')}")

# Test 1: Travel in Bhopal
test_interactive_query("is this the perfect weather traveling in Bhopal")


In [ ]:
# Test 2: Cycling in Bhopal today
test_interactive_query("Is it safe to cycle in Bhopal today?")


In [ ]:
# Test 3: Uncovered Activity (Honest Boundary Refusal)
test_interactive_query("Is it safe to go scuba diving in the Andamans?")


## 3. Systematic Verification of Assignment Test Cases
Here we execute each of the required test scenarios defined in the assignment specification.


In [ ]:
# Scenario 1: High Wind & Two-Wheeler / Cycling Safety (SOP-001)
import weather

mock_wind_facts = {
    "temperature_2m": 25.0,
    "apparent_temperature": 25.0,
    "wind_speed_10m": 38.0,
    "wind_gusts_10m": 52.0,
    "precipitation": 0.0,
    "precipitation_probability": 0.0,
    "uv_index": 3.0,
    "weather_code": 1,
    "pressure_msl": 1010.0,
    "relative_humidity_2m": 40.0,
    "daily_precipitation_sum": 0.0,
    "precipitation_probability_max": 0.0,
    "wind_gusts_10m_max": 52.0,
    "uv_index_max": 3.0,
    "time": "2026-11-13T12:00",
    "hour": 12,
    "location_name": "Bhopal, India",
}

orig_extract = weather.extract_weather_facts
try:
    weather.extract_weather_facts = lambda *args, **kwargs: dict(mock_wind_facts)
    res = graph.ask_bot("Is it safe to ride my scooter in Bhopal?")
    print("Query: Is it safe to ride my scooter in Bhopal?")
    print("Chosen SOP:", res.get('chosen_sop', {}).get('id'))
    print("Reply:", res.get('reply'))
finally:
    weather.extract_weather_facts = orig_extract


In [ ]:
# Scenario 2: Peak UV Index Midday Rule (SOP-002)
mock_uv_facts = {
    "temperature_2m": 32.0,
    "apparent_temperature": 34.0,
    "wind_speed_10m": 12.0,
    "wind_gusts_10m": 18.0,
    "precipitation": 0.0,
    "precipitation_probability": 0.0,
    "uv_index": 9.5,
    "weather_code": 0,
    "pressure_msl": 1012.0,
    "relative_humidity_2m": 35.0,
    "daily_precipitation_sum": 0.0,
    "precipitation_probability_max": 0.0,
    "wind_gusts_10m_max": 18.0,
    "uv_index_max": 9.5,
    "time": "2026-11-13T13:00",
    "hour": 13,
    "location_name": "Delhi, India",
}

try:
    weather.extract_weather_facts = lambda *args, **kwargs: dict(mock_uv_facts)
    res = graph.ask_bot("Should I go for a run at noon in Delhi?")
    print("Query: Should I go for a run at noon in Delhi?")
    print("Chosen SOP:", res.get('chosen_sop', {}).get('id'))
    print("Reply:", res.get('reply'))
finally:
    weather.extract_weather_facts = orig_extract


In [ ]:
# Scenario 3: Monsoon Heavy Rain Depression Proxy (SOP-SYSTEM-RAIN)
# Real-world scenario: Severe MP monsoon system with daily precipitation >= 64.5 mm
mock_monsoon_facts = {
    "temperature_2m": 22.0,
    "apparent_temperature": 22.0,
    "wind_speed_10m": 30.0,
    "wind_gusts_10m": 45.0,
    "precipitation": 15.0,
    "precipitation_probability": 95.0,
    "uv_index": 1.0,
    "weather_code": 65,
    "pressure_msl": 995.0,
    "relative_humidity_2m": 98.0,
    "daily_precipitation_sum": 85.0,
    "precipitation_probability_max": 100.0,
    "wind_gusts_10m_max": 50.0,
    "uv_index_max": 2.0,
    "time": "2026-11-13T10:00",
    "hour": 10,
    "location_name": "Bhopal, India",
}

try:
    weather.extract_weather_facts = lambda *args, **kwargs: dict(mock_monsoon_facts)
    res = graph.ask_bot("Is it safe to drive to work in Bhopal today?")
    print("Query: Is it safe to drive to work in Bhopal today?")
    print("Chosen SOP (Global Override):", res.get('chosen_sop', {}).get('id'))
    print("Selection Reason:", res.get('selection_reason'))
    print("Reply:", res.get('reply'))
finally:
    weather.extract_weather_facts = orig_extract


## 4. Multi-turn Session Memory Verification
Verify that location and activity context persists cleanly across turns using LangGraph MemorySaver checkpointer.


In [ ]:
thread_session = "colab_session_" + str(int(time.time()))

# Turn 1: Specify location and activity
res1 = graph.ask_bot("Is it safe to cycle in Bhopal?", thread_id=thread_session)
print("Turn 1 Reply:", res1.get('reply')[:120], "...")
print("Turn 1 Resolved Location:", res1.get('location'))

# Turn 2: Follow-up query without mentioning location
res2 = graph.ask_bot("What about going for a run this evening?", thread_id=thread_session)
print("\nTurn 2 Reply:", res2.get('reply')[:120], "...")
print("Turn 2 Inherited Location:", res2.get('location'))
assert res2.get('location') == 'Bhopal, India' or 'Bhopal' in str(res2.get('location'))
print("\n Memory test PASSED: Location correctly persisted across turns!")


## 5. Live Dynamic SOP Hot-Reload Test
Demonstrates adding a new 17th SOP live to `sops.yaml` without touching control-flow code!


In [ ]:
import yaml

sops_path = Path('sops.yaml')
with open(sops_path, 'r', encoding='utf-8') as f:
    orig_sops = yaml.safe_load(f)

new_test_sop = {
    "id": "SOP-999",
    "title": "Extreme Monsoon Flash Flood Warning",
    "category": "travel",
    "severity": 4,
    "type": "numeric",
    "applies_to": ["travel", "driving", "commute"],
    "condition": "daily_precipitation_sum >= 1.0",
    "advice": "URGENT: Flash flood protocol active in {location_name}. Avoid travel completely. Policy: SOP-999.",
    "rationale": "Extreme flood hazard to motorists.",
}

try:
    with open(sops_path, 'w', encoding='utf-8') as f:
        yaml.safe_dump(orig_sops + [new_test_sop], f)
    print("Temporarily appended SOP-999 to sops.yaml...")
    
    mock_rain = dict(mock_wind_facts)
    mock_rain["daily_precipitation_sum"] = 5.0
    weather.extract_weather_facts = lambda *args, **kwargs: mock_rain
    
    res = graph.ask_bot("Can I drive to work in Bhopal today?")
    print("Chosen SOP:", res.get('chosen_sop', {}).get('id'))
    assert res.get('chosen_sop', {}).get('id') == "SOP-999"
    print("\n Zero-Code Hot-Reload PASSED: New rule immediately enforced on the next turn!")
finally:
    with open(sops_path, 'w', encoding='utf-8') as f:
        yaml.safe_dump(orig_sops, f)
    weather.extract_weather_facts = orig_extract
    print("Restored original sops.yaml.")


## 6. Automated Evaluation Suite Execution & Results Table
Run the complete automated evaluation suite and view the formatted summary DataFrame.


In [ ]:
import evals

print('Running full automated test suite...')
evals.EVAL_RESULTS.clear()
evals.run_all_evals()

df = pd.DataFrame(evals.EVAL_RESULTS)
df = df.rename(columns={
    'name': 'Test Case',
    'check': 'Target Evaluation',
    'criteria': 'Pass Criteria',
    'status': 'Status',
    'notes': 'Observation Notes'
})

print(f"\nTOTAL TESTS: {len(df)} | PASSED: {(df['Status'] == 'PASS').sum()} | FAILED: {(df['Status'] == 'FAIL').sum()}")
df[['Test Case', 'Status', 'Pass Criteria', 'Observation Notes']]
